PROCESSING RAW DATA

In [3]:
import os
import glob
import pandas as pd

# Process and split the data into 2 datasets: one for each hospital.
def process_and_save_hospital(folder_name, output_filename):
    print(f"Processing {folder_name}...")

    # Find all .psv files in the folder
    search_path = os.path.join('../data/raw/physionet.org/files/challenge-2019/1.0.0/training/', folder_name, '*.psv')
    all_files = glob.glob(search_path)

    # Loop through each file, add ID/Time columns, and concatenate them into a single DataFrame
    df_list = []

    for file in all_files:
        # Read the .psv file
        df = pd.read_csv(file, sep='|')

        #Extract the patient ID from the filename
        df['Patient_ID'] = os.path.basename(file).split('.')[0]

        # Add a Time column (it is specified that each line represents an hour of data)
        df['ICU_Hour'] = range(1, len(df) + 1)

        df_list.append(df)

    # Combine the dataframes into one and save it into a parquet file
    full_df = pd.concat(df_list, ignore_index=True)
    out_path = os.path.join(f'../data/processed/{output_filename}.parquet')
    full_df.to_parquet(out_path, index=False)

    print(f"Saved {output_filename}! Shape: {full_df.shape}\n")

process_and_save_hospital('training_setA', 'train_hospital_A')
process_and_save_hospital('training_setB', 'test_hospital_B')

Processing training_setA...
Saved train_hospital_A! Shape: (790215, 43)

Processing training_setB...
Saved test_hospital_B! Shape: (761995, 43)



Now that we split the raw data into train_hospital_A and test_hospital_B we will start DATA CLEANING (FILLING MISSING VALUES) for train_hospital_A data ONLY, in order to avoid potential data leakage.

We will save hospital A medians and use them to fill hospital B gaps to prevent data leakage and simulate real world conditions

Since EtC02 column is completely filled with Nan we will just drop it

In [5]:
import json

# Load the training data
print("Loading training data...")
df_train = pd.read_parquet('../data/processed/train_hospital_A.parquet')

# Drop EtCO2 column
df_train = df_train.dropna(axis=1, how='all')  # Drop columns that are all NaN 

# Defining our columns of interest
ignore_cols = ['Patient_ID', 'ICU_Hour', 'SepsisLabel', 'ICULOS']
feature_cols = [col for col in df_train.columns if col not in ignore_cols]

# Calculate the global medians
print("Calculating global medians...")
global_medians = df_train[feature_cols].median()

# Save the medians so we can use them for Hospital B later
global_medians.to_json('../data/cleaned/hospital_A_medians.json', orient='index')
print("Saved medians to hospital_A_medians.json")

# Forward-Fill missing values (Patient by Patient)
print("Forward-filling missing values patient by patient...")
df_train[feature_cols] = df_train.groupby('Patient_ID')[feature_cols].ffill()

# Global Median Fill for any remaining missing values
print("Filling remaining missing values with global medians...")
df_train[feature_cols] = df_train[feature_cols].fillna(global_medians)

# Save the cleaned training data
print("Saving cleaned training data...")
out_path = '../data/cleaned/train_hospital_A_clean.parquet'
df_train.to_parquet(out_path, index=False)

# Verify no missing data is left
missing_left = df_train.isnull().sum().sum()
print(f"Done! Total missing values left: {missing_left}")

Loading training data...
Calculating global medians...
Saved medians to hospital_A_medians.json
Forward-filling missing values patient by patient...
Filling remaining missing values with global medians...
Saving cleaned training data...
Done! Total missing values left: 0
